# FleetVision Phase 03.5：Colab Auto Review Prelabeler

目的：使用 CLIP zero-shot classification 對 `image_review_labels.csv` 產生人工審查建議值。

注意：這份 notebook 只輸出 `auto_review_suggestions.csv`，不要直接把預測結果視為 `reviewed`。


In [ ]:
# Colab setup
!pip install -q transformers pillow pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# TODO: adjust this path to your Google Drive FleetVision folder.
PROJECT_ROOT = Path('/content/drive/MyDrive/FleetVision')
INPUT_CSV = PROJECT_ROOT / 'dataset/00_catalog/image_review_labels.csv'
OUTPUT_CSV = PROJECT_ROOT / 'outputs/metadata/auto_review_suggestions.csv'

# Use a small number first for pilot. Set to None for the full dataset.
MAX_IMAGES = 500
BATCH_SIZE = 16

assert INPUT_CSV.exists(), f'Missing input CSV: {INPUT_CSV}'
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('INPUT_CSV:', INPUT_CSV)
print('OUTPUT_CSV:', OUTPUT_CSV)


In [ ]:
import pandas as pd

df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False, encoding='utf-8-sig')
if MAX_IMAGES is not None:
    df = df.head(MAX_IMAGES).copy()
print(df.shape)
df[['image_id', 'filename', 'original_path']].head()


In [ ]:
import torch
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'openai/clip-vit-base-patch32'
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)
model.eval()
print('device:', device)


In [ ]:
PHOTO_TYPE_PROMPTS = {
    'exterior': [
        'a photo of the exterior of a car',
        'a vehicle exterior inspection photo',
    ],
    'interior': [
        'a photo of the interior of a car',
        'a vehicle cabin or dashboard photo',
    ],
    'low_quality': [
        'a blurry low quality unusable car photo',
        'an overexposed dark or unreadable photo',
    ],
    'irrelevant': [
        'a photo that is not relevant to car inspection',
        'a non car irrelevant image',
    ],
}

ANGLE_PROMPTS = {
    'front': ['front view of a car'],
    'rear': ['rear view of a car'],
    'left': ['left side view of a car'],
    'right': ['right side view of a car'],
    'front_left': ['front left three quarter view of a car'],
    'front_right': ['front right three quarter view of a car'],
    'rear_left': ['rear left three quarter view of a car'],
    'rear_right': ['rear right three quarter view of a car'],
    'unknown': ['unclear vehicle angle or ambiguous car view'],
}

DAMAGE_PROMPTS = {
    '1': ['a car exterior photo with visible body damage scratches dents cracks'],
    '0': ['a car exterior photo with no visible damage'],
}

SEVERITY_PROMPTS = {
    'none': ['a car exterior photo with no visible damage'],
    'minor': ['a car exterior photo with a minor scratch or small dent'],
    'moderate': ['a car exterior photo with moderate visible damage'],
    'severe': ['a car exterior photo with severe crash damage or large body damage'],
}


In [ ]:
def flatten_prompts(prompt_map):
    labels = []
    prompts = []
    for label, label_prompts in prompt_map.items():
        for prompt in label_prompts:
            labels.append(label)
            prompts.append(prompt)
    return labels, prompts

@torch.no_grad()
def encode_text(prompt_map):
    labels, prompts = flatten_prompts(prompt_map)
    inputs = processor(text=prompts, return_tensors='pt', padding=True, truncation=True).to(device)
    features = model.get_text_features(**inputs)
    features = features / features.norm(dim=-1, keepdim=True)
    return labels, features

PHOTO_LABELS, PHOTO_TEXT = encode_text(PHOTO_TYPE_PROMPTS)
ANGLE_LABELS, ANGLE_TEXT = encode_text(ANGLE_PROMPTS)
DAMAGE_LABELS, DAMAGE_TEXT = encode_text(DAMAGE_PROMPTS)
SEVERITY_LABELS, SEVERITY_TEXT = encode_text(SEVERITY_PROMPTS)

@torch.no_grad()
def classify_images(images, labels, text_features):
    inputs = processor(images=images, return_tensors='pt').to(device)
    image_features = model.get_image_features(**inputs)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    logits = image_features @ text_features.T
    probs = logits.softmax(dim=-1).cpu()

    rows = []
    for prob in probs:
        by_label = {}
        for label, score in zip(labels, prob.tolist()):
            by_label[label] = max(by_label.get(label, 0.0), float(score))
        best_label, best_score = max(by_label.items(), key=lambda item: item[1])
        rows.append((best_label, best_score))
    return rows

def resolve_image_path(original_path):
    path = Path(original_path)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path

def load_pil(path):
    with Image.open(path) as img:
        return img.convert('RGB')


In [ ]:
records = []

for start in tqdm(range(0, len(df), BATCH_SIZE)):
    batch = df.iloc[start:start + BATCH_SIZE].copy()
    images = []
    valid_rows = []

    for _, row in batch.iterrows():
        image_path = resolve_image_path(row['original_path'])
        if not image_path.exists():
            records.append({
                'image_id': row.get('image_id', ''),
                'filename': row.get('filename', ''),
                'original_path': row.get('original_path', ''),
                'suggested_photo_type_review': 'unknown',
                'photo_type_confidence': 0.0,
                'suggested_angle_review': 'unknown',
                'angle_confidence': 0.0,
                'suggested_has_visible_damage_review': '',
                'damage_confidence': 0.0,
                'suggested_severity_review': 'unknown',
                'severity_confidence': 0.0,
                'auto_review_notes': f'missing_image:{image_path}',
            })
            continue
        try:
            images.append(load_pil(image_path))
            valid_rows.append(row)
        except Exception as exc:
            records.append({
                'image_id': row.get('image_id', ''),
                'filename': row.get('filename', ''),
                'original_path': row.get('original_path', ''),
                'suggested_photo_type_review': 'low_quality',
                'photo_type_confidence': 0.0,
                'suggested_angle_review': 'unknown',
                'angle_confidence': 0.0,
                'suggested_has_visible_damage_review': '',
                'damage_confidence': 0.0,
                'suggested_severity_review': 'unknown',
                'severity_confidence': 0.0,
                'auto_review_notes': f'image_load_error:{exc}',
            })

    if not images:
        continue

    photo_preds = classify_images(images, PHOTO_LABELS, PHOTO_TEXT)
    angle_preds = classify_images(images, ANGLE_LABELS, ANGLE_TEXT)
    damage_preds = classify_images(images, DAMAGE_LABELS, DAMAGE_TEXT)
    severity_preds = classify_images(images, SEVERITY_LABELS, SEVERITY_TEXT)

    for row, photo_pred, angle_pred, damage_pred, severity_pred in zip(valid_rows, photo_preds, angle_preds, damage_preds, severity_preds):
        photo_label, photo_conf = photo_pred
        angle_label, angle_conf = angle_pred
        damage_label, damage_conf = damage_pred
        severity_label, severity_conf = severity_pred

        quality_status = str(row.get('quality_status', '')).strip().lower()
        if quality_status in {'low_quality', 'unreadable', 'bad'}:
            photo_label, photo_conf = 'low_quality', 1.0
            angle_label, angle_conf = 'unknown', 1.0
            damage_label, damage_conf = '', 0.0
            severity_label, severity_conf = 'unknown', 1.0

        if photo_label != 'exterior':
            angle_label = 'unknown'
            angle_conf = 1.0
            damage_label = '0'
            damage_conf = max(damage_conf, 0.5)
            severity_label = 'none'
            severity_conf = max(severity_conf, 0.5)

        records.append({
            'image_id': row.get('image_id', ''),
            'filename': row.get('filename', ''),
            'original_path': row.get('original_path', ''),
            'suggested_photo_type_review': photo_label,
            'photo_type_confidence': round(float(photo_conf), 6),
            'suggested_angle_review': angle_label,
            'angle_confidence': round(float(angle_conf), 6),
            'suggested_has_visible_damage_review': damage_label,
            'damage_confidence': round(float(damage_conf), 6),
            'suggested_severity_review': severity_label,
            'severity_confidence': round(float(severity_conf), 6),
            'auto_review_notes': 'clip_zero_shot_prelabel',
        })

suggestions = pd.DataFrame(records)
suggestions.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print('saved:', OUTPUT_CSV)
print(suggestions.shape)
suggestions.head()


In [ ]:
# Quick distribution check
for col in ['suggested_photo_type_review', 'suggested_angle_review', 'suggested_has_visible_damage_review', 'suggested_severity_review']:
    print('\n', col)
    print(suggestions[col].value_counts(dropna=False).head(20))
